# Thesis post-processing notebook — professor-requested additions

**Purpose.** This notebook collects and organizes the post-processing work added after the supervisor feedback.  
It is intentionally separated from model training: **no model is retrained here** and no expensive FOM dataset is recomputed.

The notebook addresses the requested additions:

1. physical visualizations of computed solutions;
2. time-resolved error curves;
3. deeper comparisons among the implemented strategies;
4. validation on the additional stochastic heat benchmark;
5. POD diagnostics and exact rank-selection verification;
6. preparation of final thesis figures for Overleaf.

The two benchmarks are:

- **RAD** — randomized advection–diffusion;
- **Heat** — stochastic heat equation.

Primary generative strategies:

- Conditional RealNVP;
- Static Conditional Flow Matching;
- Recurrent Residual Flow Matching.

> Run the notebook from top to bottom.  
> Cells marked **optional / recovery only** can be skipped when the required file already exists on Google Drive.


## Chapter 0 — Runtime conventions and reproducibility notes

Important interpretation rules used throughout the notebook:

- generated trajectories are **not paired** with individual FOM realizations;
- trajectory energy discrepancies are **not percentages**;
- negative finite-sample energy-discrepancy estimates are kept unchanged because the implemented estimator is unbiased;
- the empirical 10–90% bands shown later are **variability bands, not confidence intervals**;
- reported online times measure **surrogate generation only**, not FOM speedup;
- the recurrent model is a **discrete-time latent transition generator**, not an identified neural SDE;
- for RAD, long-horizon degradation is described as *consistent with* autoregressive error accumulation, not as proof of causality.


# Chapter 1 — Drive, repository and result recovery

This chapter reconstructs the lightweight working environment after a Colab reset.

The expensive scientific outputs are stored on Google Drive.  
The Git bundle restores the exact source-code revision used for the final experiments.


In [ ]:
# ============================================================
# 1.1 — Mount Google Drive and define all project paths
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_TESI = Path("/content/drive/MyDrive/Tesi")

REPO_DIR = Path("/content/tesi-work/generative-spde-surrogates")
SRC_DIR = REPO_DIR / "src"

RAD_OUT = DRIVE_TESI / "completamento_dinamico" / "rad_40x10"
HEAT_OUT = DRIVE_TESI / "completamento_dinamico" / "heat_40x10"

FIG_ROOT = DRIVE_TESI / "figure_finali_tesi"
RAD_FIG = FIG_ROOT / "rad"
HEAT_FIG = FIG_ROOT / "heat"

RAD_FIG.mkdir(parents=True, exist_ok=True)
HEAT_FIG.mkdir(parents=True, exist_ok=True)

BACKUP_DIR = DRIVE_TESI / "backup_codice_finale"
BACKUP_BUNDLE = (
    BACKUP_DIR
    / "generative-spde-surrogates_23ab4cf_20261012_153830_UTC.bundle"
)

EXPECTED_COMMIT = "23ab4cf49121eb4002ff8b7a53ba68dba5b91499"

print("=== MAIN PATHS ===")
print("Repository:", REPO_DIR)
print("RAD output:", RAD_OUT)
print("Heat output:", HEAT_OUT)
print("Final figures:", FIG_ROOT)
print("Git bundle exists:", BACKUP_BUNDLE.is_file())


In [ ]:
# ============================================================
# 1.2 — Restore the exact repository revision from the Git bundle
# Recovery-only cell: safe to rerun after a fresh Colab runtime.
# ============================================================

import subprocess
import shutil

def run(cmd, cwd=None):
    result = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        capture_output=True,
        check=True,
    )
    return result.stdout.strip()

needs_restore = True

if (REPO_DIR / ".git").is_dir():
    try:
        current_commit = run(
            ["git", "rev-parse", "HEAD"],
            cwd=REPO_DIR,
        )
        needs_restore = current_commit != EXPECTED_COMMIT
        print("Existing repository commit:", current_commit)
    except Exception:
        needs_restore = True

if needs_restore:
    print("Restoring repository from Git bundle...")

    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)

    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

    run([
        "git",
        "clone",
        str(BACKUP_BUNDLE),
        str(REPO_DIR),
    ])

    # The bundle contains the final thesis branch.
    branches = run(
        ["git", "branch", "-a"],
        cwd=REPO_DIR,
    )
    print(branches)

    run(
        ["git", "checkout", "thesis/dynamic-completion"],
        cwd=REPO_DIR,
    )

current_commit = run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
)

print("\n=== REPOSITORY CHECK ===")
print("Current commit:", current_commit)
print("Expected commit:", EXPECTED_COMMIT)
print("Exact match:", current_commit == EXPECTED_COMMIT)
print("src exists:", SRC_DIR.is_dir())
print("Python files:", len(list(SRC_DIR.rglob("*.py"))))


In [ ]:
# ============================================================
# 1.3 — Restore the RAD dataset path expected by the configuration
# No 800 MB copy is performed: a symbolic link points to Drive.
# ============================================================

RAD_DATA_DRIVE = (
    DRIVE_TESI
    / "generative-spde-surrogates"
    / "data"
    / "static_comparison_40x10.npz"
)

RAD_DATA_EXPECTED = (
    REPO_DIR
    / "data"
    / "static_comparison_40x10.npz"
)

RAD_DATA_EXPECTED.parent.mkdir(parents=True, exist_ok=True)

print("Drive RAD dataset exists:", RAD_DATA_DRIVE.is_file())

if RAD_DATA_EXPECTED.is_symlink():
    RAD_DATA_EXPECTED.unlink()
elif RAD_DATA_EXPECTED.exists():
    print("Expected path already contains a real file; leaving it unchanged.")
else:
    RAD_DATA_EXPECTED.symlink_to(RAD_DATA_DRIVE)

print("Expected path exists:", RAD_DATA_EXPECTED.exists())
print("Is symbolic link:", RAD_DATA_EXPECTED.is_symlink())

if RAD_DATA_EXPECTED.is_symlink():
    print("Points to:", RAD_DATA_EXPECTED.resolve())


In [ ]:
# ============================================================
# 1.4 — Inventory the stored evaluation outputs
# ============================================================

def inventory_case(case_name, output_dir):
    eval_dir = output_dir / "evaluation_test"

    print("\n" + "=" * 72)
    print(case_name)
    print("=" * 72)
    print("Evaluation directory:", eval_dir)
    print("Exists:", eval_dir.is_dir())

    if not eval_dir.is_dir():
        return

    files = sorted(p for p in eval_dir.iterdir() if p.is_file())
    print("Number of files:", len(files))

    for suffix in [".png", ".npz", ".json", ".csv", ".tex", ".md"]:
        subset = [p.name for p in files if p.suffix.lower() == suffix]
        if subset:
            print(f"\n{suffix} files ({len(subset)}):")
            for name in subset:
                print("  ", name)

inventory_case("RAD", RAD_OUT)
inventory_case("STOCHASTIC HEAT", HEAT_OUT)


In [ ]:
# ============================================================
# 1.5 — Copy already-generated evaluation figures to the final thesis folder
# ============================================================

import shutil

def copy_existing_pngs(prefix, output_dir, destination):
    eval_dir = output_dir / "evaluation_test"
    destination.mkdir(parents=True, exist_ok=True)

    copied = []

    for src in sorted(eval_dir.glob("*.png")):
        dst = destination / f"{prefix}_{src.name}"
        shutil.copy2(src, dst)
        copied.append(dst.name)

    print(f"\n{prefix.upper()} copied figures ({len(copied)}):")
    for name in copied:
        print("  ", name)

copy_existing_pngs("rad", RAD_OUT, RAD_FIG)
copy_existing_pngs("heat", HEAT_OUT, HEAT_FIG)


# Chapter 2 — Load POD data, FOM datasets and generated latent trajectories

The following cells load all data required by the post-processing workflow.

Expected dimensions:

- RAD FOM: `(400, 101, 2763)`, POD rank `79`;
- Heat FOM: `(400, 101, 129)`, POD rank `6`;
- generated ensemble per test group: `100` trajectories.


In [ ]:
# ============================================================
# 2.1 — Import project utilities and inspect both benchmarks
# ============================================================

import sys
import importlib
import yaml
import numpy as np

src_string = str(SRC_DIR.resolve())
if src_string not in sys.path:
    sys.path.insert(0, src_string)

importlib.invalidate_caches()

from spde_surrogates.completion.common import as_numpy
from spde_surrogates.completion.data import load_prepared, validate_dataset
from spde_surrogates.data.dataset import load_fom_dataset

print("Project imports: OK")

RAD_CONFIG = RAD_OUT / "config_colab.yaml"
HEAT_CONFIG = HEAT_OUT / "config_heat_colab.yaml"

def inspect_case(case_name, config_path, output_dir):
    print("\n" + "=" * 70)
    print(case_name)
    print("=" * 70)

    with config_path.open(encoding="utf-8") as stream:
        cfg = yaml.safe_load(stream)

    prepared = load_prepared(cfg)

    print("\n--- PREPARED DATA ---")
    print("POD modes:", prepared["pod"]["n_modes"])
    print("POD basis shape:", tuple(as_numpy(prepared["pod"]["basis"]).shape))
    print("POD mean shape:", tuple(as_numpy(prepared["pod"]["mean"]).shape))
    print("Reduced coefficients shape:", tuple(as_numpy(prepared["coefficients"]).shape))

    dataset = load_fom_dataset(cfg["data"]["path"])
    validate_dataset(dataset, cfg)

    mu_bank, times, mus, trajectories, groups = dataset

    print("\n--- FOM DATASET ---")
    print("Parameter bank shape:", np.asarray(mu_bank).shape)
    print("Times shape:", np.asarray(times).shape)
    print("Conditions shape:", np.asarray(mus).shape)
    print("Trajectories shape:", np.asarray(trajectories).shape)
    print("Groups shape:", np.asarray(groups).shape)
    print("Physical time interval:", float(times[0]), "->", float(times[-1]))

    test_ids = as_numpy(prepared["split"]["test_indices"]).astype(int)
    test_groups = sorted(
        int(g)
        for g in np.unique(np.asarray(groups)[test_ids])
    )

    print("\n--- TEST SPLIT ---")
    print("Test groups:", test_groups)
    print("Number of test trajectories:", len(test_ids))

    first_group = test_groups[0]

    model_files = {
        "RealNVP": f"realnvp_shared_s123_g{first_group}.npz",
        "Static FM": f"flow_matching_shared_s123_g{first_group}.npz",
        "Recurrent FM": f"dynamic_s123_g{first_group}.npz",
    }

    print("\n--- GENERATED FILES ---")
    for model_name, filename in model_files.items():
        path = output_dir / "evaluation_test" / filename
        print(f"\n{model_name}:")
        print("File exists:", path.is_file())

        if path.is_file():
            with np.load(path) as data:
                print("Keys:", list(data.files))
                for key in data.files:
                    arr = data[key]
                    print(f"  {key}: shape={arr.shape}, dtype={arr.dtype}")

    return {
        "config": cfg,
        "prepared": prepared,
        "dataset": dataset,
        "test_groups": test_groups,
    }

RAD_CASE = inspect_case(
    "RAD BENCHMARK",
    RAD_CONFIG,
    RAD_OUT,
)

HEAT_CASE = inspect_case(
    "STOCHASTIC HEAT BENCHMARK",
    HEAT_CONFIG,
    HEAT_OUT,
)


# Chapter 3 — Stochastic heat equation: physical-space visualizations

This chapter addresses the request for **visualizations of computed solutions** on the additional benchmark.

A fixed test condition is used throughout:

- test group `5`;
- stored run / evaluation seed `123`;
- sample index `0`.

The synthetic realization is **not expected to match the selected FOM realization pointwise**.  
The comparison is qualitative at the single-sample level and distributional at the ensemble level.


In [ ]:
# ============================================================
# 3.1 — Reconstruct Heat generated trajectories in physical space
# ============================================================

HEAT_GROUP = 5
HEAT_SEED = 123
HEAT_SAMPLE_INDEX = 0

heat_prepared = HEAT_CASE["prepared"]

heat_basis = np.asarray(
    as_numpy(heat_prepared["pod"]["basis"])
)
heat_mean = np.asarray(
    as_numpy(heat_prepared["pod"]["mean"])
).reshape(-1)

(
    heat_mu_bank,
    heat_times,
    heat_mus,
    heat_trajectories,
    heat_groups,
) = HEAT_CASE["dataset"]

heat_times = np.asarray(heat_times)
heat_mus = np.asarray(heat_mus)
heat_trajectories = np.asarray(heat_trajectories)
heat_groups = np.asarray(heat_groups)

heat_reference_ids = np.flatnonzero(heat_groups == HEAT_GROUP)
heat_reference_fields = heat_trajectories[heat_reference_ids]
heat_reference_mu = heat_mus[heat_reference_ids[0]]

def reconstruct_heat_model(filename):
    path = HEAT_OUT / "evaluation_test" / filename

    with np.load(path) as data:
        coefficients = np.asarray(data["coefficients"])
        model_times = np.asarray(data["times"])
        model_mu = np.asarray(data["mu"])

    fields = (
        coefficients @ heat_basis
        + heat_mean[None, None, :]
    )

    return {
        "coefficients": coefficients,
        "fields": fields,
        "times": model_times,
        "mu": model_mu,
    }

heat_realnvp = reconstruct_heat_model(
    f"realnvp_shared_s{HEAT_SEED}_g{HEAT_GROUP}.npz"
)
heat_static_fm = reconstruct_heat_model(
    f"flow_matching_shared_s{HEAT_SEED}_g{HEAT_GROUP}.npz"
)
heat_recurrent_fm = reconstruct_heat_model(
    f"dynamic_s{HEAT_SEED}_g{HEAT_GROUP}.npz"
)

print("=== POD INFORMATION ===")
print("Basis shape:", heat_basis.shape)
print("Mean shape:", heat_mean.shape)

print("\n=== REFERENCE DATA ===")
print("Reference trajectories:", heat_reference_fields.shape)
print("Physical parameter mu:", heat_reference_mu)

print("\n=== GENERATED PHYSICAL FIELDS ===")
for name, model in [
    ("RealNVP", heat_realnvp),
    ("Static FM", heat_static_fm),
    ("Recurrent FM", heat_recurrent_fm),
]:
    print(name, model["fields"].shape)
    print("  time grid identical:", np.allclose(model["times"], heat_times))
    print("  mu identical:", np.allclose(model["mu"], heat_reference_mu))

# Impose the known deterministic initial field exactly.
heat_exact_initial = heat_reference_fields[0, 0, :].copy()

for model in [heat_realnvp, heat_static_fm, heat_recurrent_fm]:
    model["fields"][:, 0, :] = heat_exact_initial[None, :]

print("\n=== EXACT INITIAL CONDITION CHECK ===")
for name, model in [
    ("RealNVP", heat_realnvp),
    ("Static FM", heat_static_fm),
    ("Recurrent FM", heat_recurrent_fm),
]:
    err = np.max(np.abs(model["fields"][:, 0, :] - heat_exact_initial[None, :]))
    print(f"{name}: {err:.6e}")


In [ ]:
# ============================================================
# 3.2 — Heat fixed-sample space-time visualization
# ============================================================

import matplotlib.pyplot as plt

x_heat = np.linspace(0.0, 1.0, heat_reference_fields.shape[-1])

heat_fixed_samples = {
    "FOM": heat_reference_fields[HEAT_SAMPLE_INDEX],
    "RealNVP": heat_realnvp["fields"][HEAT_SAMPLE_INDEX],
    "Static FM": heat_static_fm["fields"][HEAT_SAMPLE_INDEX],
    "Recurrent FM": heat_recurrent_fm["fields"][HEAT_SAMPLE_INDEX],
}

all_values = np.concatenate([
    values.reshape(-1)
    for values in heat_fixed_samples.values()
])

vmin = float(all_values.min())
vmax = float(all_values.max())

fig, axes = plt.subplots(
    1,
    4,
    figsize=(14, 4.3),
    sharey=True,
)

image_obj = None

for ax, (name, values) in zip(axes, heat_fixed_samples.items()):
    image_obj = ax.imshow(
        values,
        origin="lower",
        aspect="auto",
        extent=[
            float(x_heat[0]),
            float(x_heat[-1]),
            float(heat_times[0]),
            float(heat_times[-1]),
        ],
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(name)
    ax.set_xlabel("Spatial coordinate x")

axes[0].set_ylabel("Physical time t")

fig.colorbar(
    image_obj,
    ax=axes.ravel().tolist(),
    shrink=0.82,
    label="u(x,t)",
)

fig.suptitle(
    f"Stochastic heat equation — group {HEAT_GROUP}, "
    f"seed {HEAT_SEED}, sample index {HEAT_SAMPLE_INDEX}"
)

HEAT_FIXED_FIG = HEAT_FIG / "heat_fixed_sample_spacetime.png"

fig.savefig(
    HEAT_FIXED_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

print("Saved:", HEAT_FIXED_FIG)


In [ ]:
# ============================================================
# 3.3 — Heat ensemble mean and standard-deviation profiles
# ============================================================

import csv

HEAT_PROFILE_TIMES = [1.0, 3.0, 5.0]

heat_profile_ids = [
    int(np.argmin(np.abs(heat_times - t)))
    for t in HEAT_PROFILE_TIMES
]

heat_ensembles = {
    "FOM": heat_reference_fields,
    "RealNVP": heat_realnvp["fields"],
    "Static FM": heat_static_fm["fields"],
    "Recurrent FM": heat_recurrent_fm["fields"],
}

heat_means = {
    name: values.mean(axis=0)
    for name, values in heat_ensembles.items()
}

heat_stds = {
    name: values.std(axis=0, ddof=1)
    for name, values in heat_ensembles.items()
}

def relative_l2(prediction, reference):
    return float(
        np.linalg.norm(prediction - reference)
        / (np.linalg.norm(reference) + 1e-12)
    )

# ---- Mean figure ----
fig, axes = plt.subplots(
    1,
    len(heat_profile_ids),
    figsize=(14, 4),
    sharey=True,
)

for ax, time_id in zip(axes, heat_profile_ids):
    for name in ["FOM", "RealNVP", "Static FM", "Recurrent FM"]:
        ax.plot(x_heat, heat_means[name][time_id], label=name)

    ax.set_title(f"t = {heat_times[time_id]:.2f}")
    ax.set_xlabel("Spatial coordinate x")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Ensemble mean")
axes[-1].legend()

fig.suptitle(
    f"Stochastic heat equation — group {HEAT_GROUP}: ensemble mean"
)
fig.tight_layout()

HEAT_MEAN_FIG = HEAT_FIG / "heat_mean_profiles.png"
fig.savefig(HEAT_MEAN_FIG, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ---- Standard-deviation figure ----
fig, axes = plt.subplots(
    1,
    len(heat_profile_ids),
    figsize=(14, 4),
    sharey=True,
)

for ax, time_id in zip(axes, heat_profile_ids):
    for name in ["FOM", "RealNVP", "Static FM", "Recurrent FM"]:
        ax.plot(x_heat, heat_stds[name][time_id], label=name)

    ax.set_title(f"t = {heat_times[time_id]:.2f}")
    ax.set_xlabel("Spatial coordinate x")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Ensemble standard deviation")
axes[-1].legend()

fig.suptitle(
    f"Stochastic heat equation — group {HEAT_GROUP}: "
    "ensemble standard deviation"
)
fig.tight_layout()

HEAT_STD_FIG = HEAT_FIG / "heat_std_profiles.png"
fig.savefig(HEAT_STD_FIG, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ---- Numerical errors against the finite FOM ensemble ----
rows = []

print("=== RELATIVE ERRORS AGAINST FOM ENSEMBLE ===")

for time_id in heat_profile_ids:
    print(f"\nPhysical time t = {heat_times[time_id]:.2f}")

    for name in ["RealNVP", "Static FM", "Recurrent FM"]:
        mean_err = relative_l2(
            heat_means[name][time_id],
            heat_means["FOM"][time_id],
        )
        std_err = relative_l2(
            heat_stds[name][time_id],
            heat_stds["FOM"][time_id],
        )

        rows.append([
            float(heat_times[time_id]),
            name,
            mean_err,
            std_err,
        ])

        print(
            f"{name:<13} "
            f"mean error = {mean_err:.6f}   "
            f"std error = {std_err:.6f}"
        )

HEAT_MEAN_STD_CSV = HEAT_FIG / "heat_mean_std_errors.csv"

with HEAT_MEAN_STD_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["time", "model", "mean_relative_error", "std_relative_error"])
    writer.writerows(rows)

print("\nSaved:", HEAT_MEAN_FIG)
print("Saved:", HEAT_STD_FIG)
print("Saved:", HEAT_MEAN_STD_CSV)


In [ ]:
# ============================================================
# 3.4 — Heat exact analytical mean and variance validation
# ============================================================

# Benchmark parameters from the final configuration.
HEAT_SIGMA = 0.3
HEAT_J = 16

kappa = float(heat_reference_mu[0])
A = float(heat_reference_mu[1])

phi = np.stack(
    [
        np.sqrt(2.0) * np.sin(j * np.pi * x_heat)
        for j in range(1, HEAT_J + 1)
    ],
    axis=0,
)

exact_mean = np.zeros(
    (len(heat_times), len(x_heat)),
    dtype=float,
)

exact_variance = np.zeros_like(exact_mean)

for time_id, t in enumerate(heat_times):
    for j in range(1, HEAT_J + 1):
        lam = kappa * (j * np.pi) ** 2

        if j == 1:
            mean_amplitude = (
                A * (1.0 - np.exp(-lam * t)) / lam
            )
            exact_mean[time_id] += mean_amplitude * phi[j - 1]

        variance_amplitude = (
            (HEAT_SIGMA / j) ** 2
            * (1.0 - np.exp(-2.0 * lam * t))
            / (2.0 * lam)
        )

        exact_variance[time_id] += (
            variance_amplitude * phi[j - 1] ** 2
        )

exact_std = np.sqrt(np.maximum(exact_variance, 0.0))

print("=== HEAT PARAMETERS ===")
print("kappa =", kappa)
print("A     =", A)
print("sigma =", HEAT_SIGMA)
print("J     =", HEAT_J)

# Generated ensemble statistics.
generated_means = {
    "RealNVP": heat_realnvp["fields"].mean(axis=0),
    "Static FM": heat_static_fm["fields"].mean(axis=0),
    "Recurrent FM": heat_recurrent_fm["fields"].mean(axis=0),
}

generated_variances = {
    "RealNVP": heat_realnvp["fields"].var(axis=0, ddof=1),
    "Static FM": heat_static_fm["fields"].var(axis=0, ddof=1),
    "Recurrent FM": heat_recurrent_fm["fields"].var(axis=0, ddof=1),
}

generated_stds = {
    name: np.sqrt(np.maximum(var, 0.0))
    for name, var in generated_variances.items()
}

# ---- Exact mean figure ----
fig, axes = plt.subplots(
    1,
    len(heat_profile_ids),
    figsize=(14, 4),
    sharey=True,
)

for ax, time_id in zip(axes, heat_profile_ids):
    ax.plot(
        x_heat,
        exact_mean[time_id],
        linestyle="--",
        linewidth=2,
        label="Exact",
    )

    for name in ["RealNVP", "Static FM", "Recurrent FM"]:
        ax.plot(
            x_heat,
            generated_means[name][time_id],
            label=name,
        )

    ax.set_title(f"t = {heat_times[time_id]:.2f}")
    ax.set_xlabel("Spatial coordinate x")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Mean field")
axes[-1].legend()

fig.suptitle(
    f"Stochastic heat equation — group {HEAT_GROUP}: "
    "generated mean vs exact mean"
)
fig.tight_layout()

HEAT_EXACT_MEAN_FIG = HEAT_FIG / "heat_exact_mean_profiles.png"
fig.savefig(HEAT_EXACT_MEAN_FIG, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ---- Exact standard-deviation figure ----
fig, axes = plt.subplots(
    1,
    len(heat_profile_ids),
    figsize=(14, 4),
    sharey=True,
)

for ax, time_id in zip(axes, heat_profile_ids):
    ax.plot(
        x_heat,
        exact_std[time_id],
        linestyle="--",
        linewidth=2,
        label="Exact",
    )

    for name in ["RealNVP", "Static FM", "Recurrent FM"]:
        ax.plot(
            x_heat,
            generated_stds[name][time_id],
            label=name,
        )

    ax.set_title(f"t = {heat_times[time_id]:.2f}")
    ax.set_xlabel("Spatial coordinate x")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Standard deviation")
axes[-1].legend()

fig.suptitle(
    f"Stochastic heat equation — group {HEAT_GROUP}: "
    "generated standard deviation vs exact"
)
fig.tight_layout()

HEAT_EXACT_STD_FIG = HEAT_FIG / "heat_exact_std_profiles.png"
fig.savefig(HEAT_EXACT_STD_FIG, dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

# ---- Numerical exact-moment errors ----
selected_rows = []

print("\n=== SELECTED-TIME ERRORS AGAINST EXACT MOMENTS ===")

for time_id in heat_profile_ids:
    print(f"\nPhysical time t = {heat_times[time_id]:.2f}")

    for name in ["RealNVP", "Static FM", "Recurrent FM"]:
        mean_err = relative_l2(
            generated_means[name][time_id],
            exact_mean[time_id],
        )
        variance_err = relative_l2(
            generated_variances[name][time_id],
            exact_variance[time_id],
        )

        selected_rows.append([
            float(heat_times[time_id]),
            name,
            mean_err,
            variance_err,
        ])

        print(
            f"{name:<13} "
            f"mean error = {mean_err:.6f}   "
            f"variance error = {variance_err:.6f}"
        )

global_rows = []

print("\n=== GLOBAL SPACE-TIME ERRORS AGAINST EXACT MOMENTS ===")

for name in ["RealNVP", "Static FM", "Recurrent FM"]:
    mean_err = relative_l2(
        generated_means[name],
        exact_mean,
    )
    variance_err = relative_l2(
        generated_variances[name],
        exact_variance,
    )

    global_rows.append([
        name,
        mean_err,
        variance_err,
    ])

    print(
        f"{name:<13} "
        f"mean error = {mean_err:.6f}   "
        f"variance error = {variance_err:.6f}"
    )

SELECTED_CSV = HEAT_FIG / "heat_exact_moment_errors_selected_times.csv"
GLOBAL_CSV = HEAT_FIG / "heat_exact_moment_errors_global.csv"

with SELECTED_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["time", "model", "mean_relative_error", "variance_relative_error"])
    writer.writerows(selected_rows)

with GLOBAL_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["model", "mean_relative_error", "variance_relative_error"])
    writer.writerows(global_rows)

print("\nSaved:", HEAT_EXACT_MEAN_FIG)
print("Saved:", HEAT_EXACT_STD_FIG)
print("Saved:", SELECTED_CSV)
print("Saved:", GLOBAL_CSV)


# Chapter 4 — RAD geometry recovery and physical-space visualizations

The RAD dataset stores 2763 field values but not the mesh coordinates.  
The original source reconstructs the domain with legacy FEniCS/DLROMs.

A validated plotting mesh is stored permanently on Drive as:

`figure_finali_tesi/rad/rad_mesh_for_plotting.npz`

**If that file already exists, skip Sections 4.1–4.4 and start from Section 4.5.**


In [ ]:
# ============================================================
# 4.1 — Check whether the validated RAD plotting mesh already exists
# ============================================================

RAD_MESH_FILE = RAD_FIG / "rad_mesh_for_plotting.npz"

print("Validated mesh file:", RAD_MESH_FILE)
print("Exists:", RAD_MESH_FILE.is_file())


### 4.2 — Optional recovery: create a legacy FEniCS environment

Run the next two cells **only if `RAD_MESH_FILE` does not exist**.

The isolated environment is required because the main Colab Python runtime does not provide legacy FEniCS/DOLFIN.


In [ ]:
# ============================================================
# 4.2 — OPTIONAL / RECOVERY ONLY
# Create an isolated legacy FEniCS + DLROMs environment
# ============================================================

import subprocess
import os
import tarfile
import urllib.request

MICROMAMBA = Path("/content/bin/micromamba")
FENICS_ENV = Path("/content/fenics-legacy")

Path("/content/bin").mkdir(parents=True, exist_ok=True)

if not MICROMAMBA.exists():
    print("Installing micromamba...")

    archive = Path("/tmp/micromamba.tar.bz2")
    urllib.request.urlretrieve(
        "https://micro.mamba.pm/api/micromamba/linux-64/latest",
        archive,
    )

    with tarfile.open(archive, "r:bz2") as tar:
        member = tar.getmember("bin/micromamba")
        member.name = "micromamba"
        tar.extract(member, path="/content/bin")

    MICROMAMBA.chmod(0o755)

print("Micromamba exists:", MICROMAMBA.exists())

if not FENICS_ENV.exists():
    print("Creating isolated FEniCS environment...")

    subprocess.run(
        [
            str(MICROMAMBA),
            "create",
            "-y",
            "-p",
            str(FENICS_ENV),
            "-c",
            "conda-forge",
            "python=3.11",
            "fenics=2019.1.0",
            "pip",
        ],
        check=True,
    )

# Install DLROMs in the isolated environment.
subprocess.run(
    [
        str(MICROMAMBA),
        "run",
        "-p",
        str(FENICS_ENV),
        "python",
        "-m",
        "pip",
        "install",
        "-q",
        "dlroms",
    ],
    check=True,
)

# Install the Python Gmsh bindings required by dlroms.geometry.
subprocess.run(
    [
        str(MICROMAMBA),
        "install",
        "-y",
        "-p",
        str(FENICS_ENV),
        "-c",
        "conda-forge",
        "python-gmsh",
    ],
    check=True,
)

test_script = r'''
import sys
import gmsh
from dlroms import *
from dlroms import geometry
import fenics
import dolfin

print("Python:", sys.version)
print("gmsh:", gmsh.__version__)
print("dlroms: OK")
print("fenics: OK")
print("dolfin:", dolfin.__version__)
'''

result = subprocess.run(
    [
        str(MICROMAMBA),
        "run",
        "-p",
        str(FENICS_ENV),
        "python",
        "-c",
        test_script,
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.stderr.strip():
    print(result.stderr)

print("Return code:", result.returncode)


In [ ]:
# ============================================================
# 4.3 — OPTIONAL / RECOVERY ONLY
# Generate the RAD mesh from the original geometry
# ============================================================

RAD_MESH_CANDIDATE = RAD_FIG / "rad_mesh_candidate.npz"

mesh_script = f'''
import numpy as np
import gmsh

from dlroms import *
from fenics import vertex_to_dof_map

print("gmsh version:", gmsh.__version__)

domain = (
    fe.rectangle((0, 0), (3, 2))
    - fe.circle((0.95, 0.70), 0.25)
    - fe.circle((1.50, 1.35), 0.25)
    - fe.circle((2.05, 0.70), 0.25)
)

mesh = fe.mesh(
    domain,
    stepsize=0.05,
)

Vh = fe.space(
    mesh,
    "CG",
    1,
)

gdim = mesh.geometry().dim()

coordinates = (
    Vh.tabulate_dof_coordinates()
    .reshape(Vh.dim(), gdim)
)

v2d = np.asarray(
    vertex_to_dof_map(Vh),
    dtype=int,
)

cells = np.asarray(
    mesh.cells(),
    dtype=int,
)

triangles = v2d[cells]

print("Vh dimension:", Vh.dim())
print("Mesh vertices:", mesh.num_vertices())
print("Mesh cells:", mesh.num_cells())
print("Coordinates:", coordinates.shape)
print("Triangles:", triangles.shape)

EXPECTED_DOF = 2763

if Vh.dim() != EXPECTED_DOF:
    raise RuntimeError(
        f"Expected {{EXPECTED_DOF}} DOFs, obtained {{Vh.dim()}}."
    )

np.savez_compressed(
    r"{RAD_MESH_CANDIDATE}",
    coordinates=coordinates,
    triangles=triangles,
)

print("Saved:", r"{RAD_MESH_CANDIDATE}")
'''

result = subprocess.run(
    [
        str(MICROMAMBA),
        "run",
        "-p",
        str(FENICS_ENV),
        "python",
        "-c",
        mesh_script,
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr.strip():
    print("=== STDERR ===")
    print(result.stderr)

print("Return code:", result.returncode)
print("Candidate mesh exists:", RAD_MESH_CANDIDATE.is_file())


In [ ]:
# ============================================================
# 4.4 — OPTIONAL / RECOVERY ONLY
# Validate the recovered mesh with a real FOM snapshot
# ============================================================

import matplotlib.tri as mtri

with np.load(RAD_MESH_CANDIDATE) as mesh_data:
    rad_coordinates_candidate = np.asarray(mesh_data["coordinates"])
    rad_triangles_candidate = np.asarray(mesh_data["triangles"])

(
    _rad_mu_bank,
    _rad_times,
    _rad_mus,
    _rad_trajectories,
    _rad_groups,
) = RAD_CASE["dataset"]

_rad_times = np.asarray(_rad_times)
_rad_mus = np.asarray(_rad_mus)
_rad_trajectories = np.asarray(_rad_trajectories)
_rad_groups = np.asarray(_rad_groups)

group_ids = np.flatnonzero(_rad_groups == 5)
trajectory_id = int(group_ids[0])
time_id = int(np.argmin(np.abs(_rad_times - 5.0)))

field = _rad_trajectories[trajectory_id, time_id]

edges = np.vstack([
    rad_triangles_candidate[:, [0, 1]],
    rad_triangles_candidate[:, [1, 2]],
    rad_triangles_candidate[:, [2, 0]],
])

edges = np.unique(np.sort(edges, axis=1), axis=0)

edge_jumps = np.abs(
    field[edges[:, 0]]
    - field[edges[:, 1]]
)

print("=== LOCAL SMOOTHNESS DIAGNOSTIC ===")
print("Median |jump|:", float(np.median(edge_jumps)))
print("Mean |jump|:", float(np.mean(edge_jumps)))
print("95th percentile |jump|:", float(np.quantile(edge_jumps, 0.95)))
print(
    "Median jump / field std:",
    float(np.median(edge_jumps) / field.std()),
)

triangulation = mtri.Triangulation(
    rad_coordinates_candidate[:, 0],
    rad_coordinates_candidate[:, 1],
    rad_triangles_candidate,
)

fig, ax = plt.subplots(figsize=(10, 6))

plot = ax.tripcolor(
    triangulation,
    field,
    shading="gouraud",
)

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(0.0, 3.0)
ax.set_ylim(0.0, 2.0)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("RAD FOM snapshot — group 5, sample 0, t = 5.00")

fig.colorbar(plot, ax=ax, label="u(x,t)")
fig.tight_layout()

CHECK_FIG = RAD_FIG / "rad_fom_snapshot_mesh_check.png"
fig.savefig(CHECK_FIG, dpi=250, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved:", CHECK_FIG)

# The recovered mesh was validated visually and numerically.
# Promote it to the permanent plotting mesh.
shutil.copy2(
    RAD_MESH_CANDIDATE,
    RAD_MESH_FILE,
)

print("Validated mesh saved as:", RAD_MESH_FILE)


In [ ]:
# ============================================================
# 4.5 — Load the validated RAD plotting mesh
# ============================================================

import matplotlib.tri as mtri

with np.load(RAD_MESH_FILE) as mesh_data:
    rad_coordinates = np.asarray(mesh_data["coordinates"])
    rad_triangles = np.asarray(mesh_data["triangles"])

rad_triangulation = mtri.Triangulation(
    rad_coordinates[:, 0],
    rad_coordinates[:, 1],
    rad_triangles,
)

print("Coordinates:", rad_coordinates.shape)
print("Triangles:", rad_triangles.shape)


In [ ]:
# ============================================================
# 4.6 — Reconstruct RAD generated trajectories in physical space
# ============================================================

rad_prepared = RAD_CASE["prepared"]

rad_basis = np.asarray(
    as_numpy(rad_prepared["pod"]["basis"])
)
rad_mean = np.asarray(
    as_numpy(rad_prepared["pod"]["mean"])
).reshape(-1)

(
    rad_mu_bank,
    rad_times,
    rad_mus,
    rad_trajectories,
    rad_groups,
) = RAD_CASE["dataset"]

rad_times = np.asarray(rad_times)
rad_mus = np.asarray(rad_mus)
rad_trajectories = np.asarray(rad_trajectories)
rad_groups = np.asarray(rad_groups)

RAD_GROUP = 5
RAD_SEED = 123

rad_reference_ids = np.flatnonzero(rad_groups == RAD_GROUP)
rad_reference_fields = rad_trajectories[rad_reference_ids]
rad_reference_mu = rad_mus[rad_reference_ids[0]]

def reconstruct_rad_model(filename):
    path = RAD_OUT / "evaluation_test" / filename

    with np.load(path) as data:
        coefficients = np.asarray(data["coefficients"])
        model_times = np.asarray(data["times"])
        model_mu = np.asarray(data["mu"])

    fields = (
        coefficients @ rad_basis
        + rad_mean[None, None, :]
    )

    return {
        "coefficients": coefficients,
        "fields": fields,
        "times": model_times,
        "mu": model_mu,
    }

rad_realnvp = reconstruct_rad_model(
    f"realnvp_shared_s{RAD_SEED}_g{RAD_GROUP}.npz"
)
rad_static_fm = reconstruct_rad_model(
    f"flow_matching_shared_s{RAD_SEED}_g{RAD_GROUP}.npz"
)
rad_recurrent_fm = reconstruct_rad_model(
    f"dynamic_s{RAD_SEED}_g{RAD_GROUP}.npz"
)

# The physical initial condition is exactly zero.
rad_exact_initial = rad_reference_fields[0, 0, :].copy()

for model in [rad_realnvp, rad_static_fm, rad_recurrent_fm]:
    model["fields"][:, 0, :] = rad_exact_initial[None, :]

print("=== GENERATED PHYSICAL FIELDS ===")
for name, model in [
    ("RealNVP", rad_realnvp),
    ("Static FM", rad_static_fm),
    ("Recurrent FM", rad_recurrent_fm),
]:
    print(name)
    print("  fields:", model["fields"].shape)
    print("  time grid identical:", np.allclose(model["times"], rad_times))
    print("  mu identical:", np.allclose(model["mu"], rad_reference_mu))
    print("  finite:", bool(np.all(np.isfinite(model["fields"]))))


In [ ]:
# ============================================================
# 4.7 — RAD fixed-sample physical snapshots
# ============================================================

RAD_SAMPLE_INDEX = 0
RAD_PLOT_TIMES = [0.5, 2.0, 3.5, 5.0]

rad_time_ids = [
    int(np.argmin(np.abs(rad_times - t)))
    for t in RAD_PLOT_TIMES
]

rad_samples = {
    "FOM": rad_reference_fields[RAD_SAMPLE_INDEX],
    "RealNVP": rad_realnvp["fields"][RAD_SAMPLE_INDEX],
    "Static FM": rad_static_fm["fields"][RAD_SAMPLE_INDEX],
    "Recurrent FM": rad_recurrent_fm["fields"][RAD_SAMPLE_INDEX],
}

print("=== FIELD RANGES AT SELECTED TIMES ===")
for name, values in rad_samples.items():
    selected = values[rad_time_ids]
    print(
        f"{name:<13} "
        f"min={selected.min():.6f}   "
        f"max={selected.max():.6f}"
    )

# Quantile-based visualization scale.
# The underlying physical fields are NOT clipped.
all_selected_values = np.concatenate([
    values[rad_time_ids].reshape(-1)
    for values in rad_samples.values()
])

rad_vmin = float(np.quantile(all_selected_values, 0.005))
rad_vmax = float(np.quantile(all_selected_values, 0.995))

model_names = list(rad_samples.keys())

fig, axes = plt.subplots(
    len(model_names),
    len(rad_time_ids),
    figsize=(15, 9),
    sharex=True,
    sharey=True,
)

last_plot = None

for row, model_name in enumerate(model_names):
    trajectory = rad_samples[model_name]

    for col, time_id in enumerate(rad_time_ids):
        ax = axes[row, col]

        last_plot = ax.tripcolor(
            rad_triangulation,
            trajectory[time_id],
            shading="gouraud",
            vmin=rad_vmin,
            vmax=rad_vmax,
        )

        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(0.0, 3.0)
        ax.set_ylim(0.0, 2.0)

        if row == 0:
            ax.set_title(f"t = {rad_times[time_id]:.2f}")

        if col == 0:
            ax.set_ylabel(f"{model_name}\n\ny")

        if row == len(model_names) - 1:
            ax.set_xlabel("x")

fig.colorbar(
    last_plot,
    ax=axes.ravel().tolist(),
    shrink=0.82,
    label="u(x,t)",
)

fig.suptitle(
    "Randomized advection-diffusion — "
    f"group {RAD_GROUP}, seed {RAD_SEED}, "
    f"sample index {RAD_SAMPLE_INDEX}",
    y=0.995,
)

RAD_SNAPSHOT_FIG = RAD_FIG / "rad_fixed_sample_snapshots.png"

fig.savefig(
    RAD_SNAPSHOT_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

print("Saved:", RAD_SNAPSHOT_FIG)


In [ ]:
# ============================================================
# 4.8 — RAD ensemble-mean spatial error maps
# Thesis-ready layout.
# ============================================================

rad_ensemble_fields = {
    "FOM": rad_reference_fields,
    "RealNVP": rad_realnvp["fields"],
    "Static FM": rad_static_fm["fields"],
    "Recurrent FM": rad_recurrent_fm["fields"],
}

rad_ensemble_means = {
    name: fields.mean(axis=0)
    for name, fields in rad_ensemble_fields.items()
}

def rad_relative_l2(prediction, reference):
    return float(
        np.linalg.norm(prediction - reference)
        / (np.linalg.norm(reference) + 1e-12)
    )

print("=== RELATIVE ENSEMBLE-MEAN ERRORS ===")

rad_mean_error_rows = []

for time_id in rad_time_ids:
    fom_mean = rad_ensemble_means["FOM"][time_id]

    print(f"\nPhysical time t = {rad_times[time_id]:.2f}")

    for name in ["RealNVP", "Static FM", "Recurrent FM"]:
        error = rad_relative_l2(
            rad_ensemble_means[name][time_id],
            fom_mean,
        )

        rad_mean_error_rows.append([
            float(rad_times[time_id]),
            name,
            error,
        ])

        print(
            f"{name:<13} "
            f"relative mean error = {error:.6f}"
        )

RAD_ERROR_TIMES = [2.0, 5.0]

rad_error_time_ids = [
    int(np.argmin(np.abs(rad_times - t)))
    for t in RAD_ERROR_TIMES
]

error_fields = {}

for time_id in rad_error_time_ids:
    fom_mean = rad_ensemble_means["FOM"][time_id]

    for name in ["RealNVP", "Static FM", "Recurrent FM"]:
        error_fields[(time_id, name)] = np.abs(
            rad_ensemble_means[name][time_id]
            - fom_mean
        )

all_errors = np.concatenate([
    field.reshape(-1)
    for field in error_fields.values()
])

error_vmax = float(np.quantile(all_errors, 0.995))

column_titles = [
    "FOM mean",
    "RealNVP error",
    "Static FM error",
    "Recurrent FM error",
]

fig, axes = plt.subplots(
    len(rad_error_time_ids),
    4,
    figsize=(17.5, 6.0),
    sharex=True,
    sharey=True,
    constrained_layout=True,
)

fom_plot = None
error_plot = None

for row, time_id in enumerate(rad_error_time_ids):
    fom_plot = axes[row, 0].tripcolor(
        rad_triangulation,
        rad_ensemble_means["FOM"][time_id],
        shading="gouraud",
        vmin=0.0,
        vmax=1.0,
    )

    for col, name in enumerate(
        ["RealNVP", "Static FM", "Recurrent FM"],
        start=1,
    ):
        error_plot = axes[row, col].tripcolor(
            rad_triangulation,
            error_fields[(time_id, name)],
            shading="gouraud",
            vmin=0.0,
            vmax=error_vmax,
        )

    axes[row, 0].set_ylabel(
        f"t = {rad_times[time_id]:.2f}\ny"
    )

for col, title in enumerate(column_titles):
    axes[0, col].set_title(
        title,
        fontsize=12,
        pad=8,
    )

for ax in axes.ravel():
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0.0, 3.0)
    ax.set_ylim(0.0, 2.0)

for ax in axes[-1, :]:
    ax.set_xlabel("x")

fig.colorbar(
    fom_plot,
    ax=axes[:, 0],
    shrink=0.88,
    pad=0.02,
    label="FOM ensemble mean",
)

fig.colorbar(
    error_plot,
    ax=axes[:, 1:],
    shrink=0.88,
    pad=0.02,
    label="Absolute ensemble-mean error",
)

fig.suptitle(
    "Randomized advection-diffusion — "
    f"group {RAD_GROUP}: ensemble-mean spatial errors",
    fontsize=14,
)

RAD_MEAN_ERROR_FIG = RAD_FIG / "rad_mean_error_maps.png"

fig.savefig(
    RAD_MEAN_ERROR_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

RAD_MEAN_ERROR_CSV = RAD_FIG / "rad_mean_errors_selected_times.csv"

with RAD_MEAN_ERROR_CSV.open("w", encoding="utf-8") as stream:
    stream.write("time,model,relative_mean_error\n")
    for row in rad_mean_error_rows:
        stream.write(f"{row[0]},{row[1]},{row[2]}\n")

print("Saved:", RAD_MEAN_ERROR_FIG)
print("Saved:", RAD_MEAN_ERROR_CSV)


# Chapter 5 — Time-resolved marginal distribution errors

This chapter directly addresses the supervisor request for **error trends over time**.

The stored `marginal_energy.npz` files contain one curve for each held-out group and stored evaluation run.

- RAD: 6 group curves per model;
- Heat: 6 groups × 3 seeds = 18 curves per model.

The shaded region is the empirical **10th–90th percentile variability band**.  
It is **not a confidence interval**.


In [ ]:
# ============================================================
# 5.1 — Load and plot time-resolved marginal energy discrepancy
# ============================================================

RAD_MARGINAL_FILE = RAD_OUT / "evaluation_test" / "marginal_energy.npz"
HEAT_MARGINAL_FILE = HEAT_OUT / "evaluation_test" / "marginal_energy.npz"

TEST_GROUPS = [5, 8, 13, 18, 32, 38]

# For RAD, "s123" is the stored run label for the historical static models.
RAD_RUN_LABEL = 123

# Heat was evaluated over three model-training/evaluation seeds.
HEAT_SEEDS = [123, 456, 789]

MODEL_SPECS = {
    "RealNVP": "realnvp_shared",
    "Static FM": "flow_matching_shared",
    "Recurrent FM": "dynamic",
}

def load_time_curves(npz_path, case_name, seeds):
    print("\n" + "=" * 70)
    print(case_name)
    print("=" * 70)

    curves = {}

    with np.load(npz_path, allow_pickle=False) as data:
        times_local = np.asarray(data["times"], dtype=float)

        for model_name, prefix in MODEL_SPECS.items():
            model_curves = []

            for seed in seeds:
                for group in TEST_GROUPS:
                    key = f"{prefix}_s{seed}_g{group}"

                    if key not in data.files:
                        raise KeyError(f"Missing key: {key}")

                    model_curves.append(
                        np.asarray(data[key], dtype=float)
                    )

            model_curves = np.stack(model_curves, axis=0)
            curves[model_name] = model_curves

            print(
                f"{model_name:<13}: "
                f"{model_curves.shape}"
            )

    return times_local, curves

rad_error_times, rad_time_curves = load_time_curves(
    RAD_MARGINAL_FILE,
    "RAD",
    seeds=[RAD_RUN_LABEL],
)

heat_error_times, heat_time_curves = load_time_curves(
    HEAT_MARGINAL_FILE,
    "STOCHASTIC HEAT",
    seeds=HEAT_SEEDS,
)

def summarize_curves(curves):
    return {
        model_name: {
            "mean": np.mean(values, axis=0),
            "p10": np.quantile(values, 0.10, axis=0),
            "p90": np.quantile(values, 0.90, axis=0),
        }
        for model_name, values in curves.items()
    }

rad_summary = summarize_curves(rad_time_curves)
heat_summary = summarize_curves(heat_time_curves)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 4.5),
)

cases = [
    (
        axes[0],
        rad_error_times,
        rad_summary,
        "Randomized advection-diffusion",
    ),
    (
        axes[1],
        heat_error_times,
        heat_summary,
        "Stochastic heat equation",
    ),
]

for ax, times_local, summary, title in cases:
    for model_name in ["RealNVP", "Static FM", "Recurrent FM"]:
        mean_curve = summary[model_name]["mean"]
        p10_curve = summary[model_name]["p10"]
        p90_curve = summary[model_name]["p90"]

        line, = ax.plot(
            times_local,
            mean_curve,
            linewidth=2,
            label=model_name,
        )

        ax.fill_between(
            times_local,
            p10_curve,
            p90_curve,
            alpha=0.15,
            color=line.get_color(),
        )

    ax.axhline(
        0.0,
        linewidth=1,
        linestyle="--",
        alpha=0.6,
    )

    ax.set_xlabel("Physical time t")
    ax.set_ylabel("Marginal energy discrepancy")
    ax.set_title(title)
    ax.grid(alpha=0.25)

axes[0].legend()

fig.suptitle(
    "Time-resolved marginal distribution error\n"
    "Mean curve with empirical 10–90% variability band",
    y=1.03,
)

fig.tight_layout()

TIME_ERROR_FIG = FIG_ROOT / "time_resolved_marginal_energy.png"

fig.savefig(
    TIME_ERROR_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

print("Saved:", TIME_ERROR_FIG)


In [ ]:
# ============================================================
# 5.2 — Selected-time numerical summary
# ============================================================

SELECTED_TIMES = [0.5, 1.0, 2.0, 3.0, 4.0, 5.0]

def print_selected_times(case_name, times_local, summary):
    print("\n" + "=" * 70)
    print(f"{case_name} — SELECTED TIMES")
    print("=" * 70)

    for requested_time in SELECTED_TIMES:
        time_id = int(
            np.argmin(np.abs(times_local - requested_time))
        )

        print(f"\nt = {times_local[time_id]:.2f}")

        for model_name in ["RealNVP", "Static FM", "Recurrent FM"]:
            mean_value = summary[model_name]["mean"][time_id]
            p10_value = summary[model_name]["p10"][time_id]
            p90_value = summary[model_name]["p90"][time_id]

            print(
                f"{model_name:<13} "
                f"mean={mean_value: .6f}   "
                f"p10={p10_value: .6f}   "
                f"p90={p90_value: .6f}"
            )

print_selected_times(
    "RAD",
    rad_error_times,
    rad_summary,
)

print_selected_times(
    "HEAT",
    heat_error_times,
    heat_summary,
)

TIME_ERROR_CSV = FIG_ROOT / "time_resolved_marginal_energy.csv"

with TIME_ERROR_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)

    writer.writerow([
        "case",
        "time",
        "model",
        "mean",
        "p10",
        "p90",
    ])

    for case_name, times_local, summary in [
        ("RAD", rad_error_times, rad_summary),
        ("Heat", heat_error_times, heat_summary),
    ]:
        for time_id, time_value in enumerate(times_local):
            for model_name in ["RealNVP", "Static FM", "Recurrent FM"]:
                writer.writerow([
                    case_name,
                    float(time_value),
                    model_name,
                    float(summary[model_name]["mean"][time_id]),
                    float(summary[model_name]["p10"][time_id]),
                    float(summary[model_name]["p90"][time_id]),
                ])

print("\nSaved:", TIME_ERROR_CSV)


In [ ]:
# ============================================================
# 5.3 — Early-vs-late temporal comparison and RAD crossover
# ============================================================

EARLY_INTERVAL = (0.05, 1.0)
LATE_INTERVAL = (3.0, 5.0)

def temporal_average(times, curve, t_start, t_end):
    mask = (
        (times >= t_start)
        & (times <= t_end)
    )

    t = times[mask]
    y = curve[mask]

    integral = np.trapezoid(y, t)

    return float(
        integral
        / (t[-1] - t[0])
    )

def summarize_interval(times, curves, interval):
    t_start, t_end = interval

    result = {}

    for model_name, values in curves.items():
        per_case = np.array([
            temporal_average(
                times,
                curve,
                t_start,
                t_end,
            )
            for curve in values
        ])

        result[model_name] = {
            "values": per_case,
            "mean": float(np.mean(per_case)),
            "p10": float(np.quantile(per_case, 0.10)),
            "p90": float(np.quantile(per_case, 0.90)),
        }

    return result

rad_early = summarize_interval(
    rad_error_times,
    rad_time_curves,
    EARLY_INTERVAL,
)
rad_late = summarize_interval(
    rad_error_times,
    rad_time_curves,
    LATE_INTERVAL,
)
heat_early = summarize_interval(
    heat_error_times,
    heat_time_curves,
    EARLY_INTERVAL,
)
heat_late = summarize_interval(
    heat_error_times,
    heat_time_curves,
    LATE_INTERVAL,
)

def print_interval_summary(case_name, early, late):
    print("\n" + "=" * 72)
    print(f"{case_name} — EARLY VS LATE")
    print("=" * 72)

    print(
        f"\nEarly interval: "
        f"{EARLY_INTERVAL[0]:.2f} to {EARLY_INTERVAL[1]:.2f}"
    )

    for model_name in ["RealNVP", "Static FM", "Recurrent FM"]:
        stats = early[model_name]
        print(
            f"{model_name:<13} "
            f"mean={stats['mean']: .6f}   "
            f"p10={stats['p10']: .6f}   "
            f"p90={stats['p90']: .6f}"
        )

    print(
        f"\nLate interval: "
        f"{LATE_INTERVAL[0]:.2f} to {LATE_INTERVAL[1]:.2f}"
    )

    for model_name in ["RealNVP", "Static FM", "Recurrent FM"]:
        stats = late[model_name]
        print(
            f"{model_name:<13} "
            f"mean={stats['mean']: .6f}   "
            f"p10={stats['p10']: .6f}   "
            f"p90={stats['p90']: .6f}"
        )

print_interval_summary("RAD", rad_early, rad_late)
print_interval_summary("HEAT", heat_early, heat_late)

# First sampled physical time at which Static FM becomes
# better than Recurrent FM in the RAD mean marginal discrepancy.
rad_static_mean = rad_summary["Static FM"]["mean"]
rad_recurrent_mean = rad_summary["Recurrent FM"]["mean"]

comparison = rad_static_mean < rad_recurrent_mean
comparison[0] = False

crossover_indices = np.flatnonzero(comparison)

print("\n" + "=" * 72)
print("RAD — STATIC FM / RECURRENT FM CROSSOVER")
print("=" * 72)

if len(crossover_indices) > 0:
    first_index = int(crossover_indices[0])

    print(
        "First sampled time where Static FM mean discrepancy "
        "< Recurrent FM mean discrepancy:"
    )
    print("t =", float(rad_error_times[first_index]))
    print("Static FM =", float(rad_static_mean[first_index]))
    print("Recurrent FM =", float(rad_recurrent_mean[first_index]))
else:
    print("No crossover found.")

EARLY_LATE_CSV = FIG_ROOT / "early_late_marginal_energy_summary.csv"

with EARLY_LATE_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)

    writer.writerow([
        "case",
        "interval",
        "model",
        "mean",
        "p10",
        "p90",
    ])

    for case_name, interval_name, summary in [
        ("RAD", "early_0.05_1", rad_early),
        ("RAD", "late_3_5", rad_late),
        ("Heat", "early_0.05_1", heat_early),
        ("Heat", "late_3_5", heat_late),
    ]:
        for model_name in ["RealNVP", "Static FM", "Recurrent FM"]:
            stats = summary[model_name]

            writer.writerow([
                case_name,
                interval_name,
                model_name,
                stats["mean"],
                stats["p10"],
                stats["p90"],
            ])

print("\nSaved:", EARLY_LATE_CSV)


# Chapter 6 — Deeper strategy comparison: accuracy, cost and paired bootstrap

This chapter addresses the request for a **deeper comparison among strategies**.

Two complementary analyses are used:

1. mean trajectory energy discrepancy vs median online generation time;
2. paired bootstrap across held-out parameter groups for the contrast  
   `Recurrent FM − Static FM`.

For the bootstrap:

- negative contrast → Recurrent FM is better;
- positive contrast → Static FM is better;
- inference is conditional on the already trained models;
- the resampling unit is the physical parameter group, **not individual snapshots**.


In [ ]:
# ============================================================
# 6.1 — Load final summary and paired-bootstrap JSON files
# ============================================================

import json

RAD_EVAL = RAD_OUT / "evaluation_test"
HEAT_EVAL = HEAT_OUT / "evaluation_test"

def load_summary(path):
    with path.open("r", encoding="utf-8") as stream:
        data = json.load(stream)

    return {
        row["model"]: row
        for row in data["summary"]
    }

def load_bootstrap(path):
    with path.open("r", encoding="utf-8") as stream:
        return json.load(stream)

rad_model_summary = load_summary(
    RAD_EVAL / "summary.json"
)
heat_model_summary = load_summary(
    HEAT_EVAL / "summary.json"
)

rad_bootstrap = load_bootstrap(
    RAD_EVAL / "paired_group_bootstrap.json"
)
heat_bootstrap = load_bootstrap(
    HEAT_EVAL / "paired_group_bootstrap.json"
)

print("RAD bootstrap:", rad_bootstrap)
print("\nHeat bootstrap:", heat_bootstrap)


In [ ]:
# ============================================================
# 6.2 — Accuracy–cost trade-off
# ============================================================

PRIMARY_MODELS = {
    "RealNVP": "realnvp_shared",
    "Static FM": "flow_matching_shared",
    "Recurrent FM": "dynamic",
}

def print_tradeoffs(case_name, summary):
    print("\n" + "=" * 72)
    print(f"{case_name} — ACCURACY / COST")
    print("=" * 72)

    for display_name, key in PRIMARY_MODELS.items():
        row = summary[key]

        print(
            f"{display_name:<13} "
            f"energy={row['mean_energy']:.6f}   "
            f"online={row['median_online_seconds']:.6f} s"
        )

    rnvp = summary["realnvp_shared"]
    fm = summary["flow_matching_shared"]
    dyn = summary["dynamic"]

    print("\nRelative online costs:")
    print(
        "Static FM / RealNVP =",
        fm["median_online_seconds"]
        / rnvp["median_online_seconds"],
    )
    print(
        "Recurrent FM / RealNVP =",
        dyn["median_online_seconds"]
        / rnvp["median_online_seconds"],
    )
    print(
        "Static FM / Recurrent FM =",
        fm["median_online_seconds"]
        / dyn["median_online_seconds"],
    )

print_tradeoffs("RAD", rad_model_summary)
print_tradeoffs("HEAT", heat_model_summary)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(12.5, 4.8),
)

cases = [
    (
        axes[0],
        "Randomized advection-diffusion",
        rad_model_summary,
    ),
    (
        axes[1],
        "Stochastic heat equation",
        heat_model_summary,
    ),
]

for ax, title, summary in cases:
    for display_name, key in PRIMARY_MODELS.items():
        row = summary[key]

        x = float(row["median_online_seconds"])
        y = float(row["mean_energy"])

        ax.scatter(
            x,
            y,
            s=90,
            label=display_name,
        )

        ax.annotate(
            display_name,
            (x, y),
            xytext=(7, 7),
            textcoords="offset points",
            fontsize=9,
        )

    ax.set_xscale("log")
    ax.set_xlabel("Median online generation time [s]")
    ax.set_ylabel("Mean trajectory energy discrepancy")
    ax.set_title(title)
    ax.grid(alpha=0.25, which="both")

axes[0].legend()

fig.suptitle(
    "Accuracy–cost trade-off of the primary generative strategies",
    y=1.02,
)
fig.tight_layout()

ACCURACY_COST_FIG = FIG_ROOT / "accuracy_cost_tradeoff.png"

fig.savefig(
    ACCURACY_COST_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

ACCURACY_COST_CSV = FIG_ROOT / "accuracy_cost_tradeoff.csv"

with ACCURACY_COST_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow([
        "case",
        "model",
        "mean_energy",
        "median_online_seconds",
    ])

    for case_name, summary in [
        ("RAD", rad_model_summary),
        ("Heat", heat_model_summary),
    ]:
        for display_name, key in PRIMARY_MODELS.items():
            row = summary[key]
            writer.writerow([
                case_name,
                display_name,
                row["mean_energy"],
                row["median_online_seconds"],
            ])

print("Saved:", ACCURACY_COST_FIG)
print("Saved:", ACCURACY_COST_CSV)


In [ ]:
# ============================================================
# 6.3 — Paired-group bootstrap figure
# ============================================================

print("=== PAIRED-GROUP BOOTSTRAP ===")

for case_name, data in [
    ("RAD", rad_bootstrap),
    ("Heat", heat_bootstrap),
]:
    mean_difference = float(data["mean_difference"])
    ci_low = float(data["ci95"][0])
    ci_high = float(data["ci95"][1])

    print(f"\n{case_name}")
    print("Recurrent FM - Static FM =", mean_difference)
    print("95% bootstrap interval =", (ci_low, ci_high))
    print("Contains zero:", ci_low <= 0.0 <= ci_high)
    print("Number of groups:", data["n_groups"])

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 3.8),
)

bootstrap_cases = [
    (
        axes[0],
        "Randomized advection-diffusion",
        rad_bootstrap,
    ),
    (
        axes[1],
        "Stochastic heat equation",
        heat_bootstrap,
    ),
]

for ax, title, data in bootstrap_cases:
    mean_difference = float(data["mean_difference"])
    ci_low = float(data["ci95"][0])
    ci_high = float(data["ci95"][1])

    lower_error = mean_difference - ci_low
    upper_error = ci_high - mean_difference

    ax.errorbar(
        mean_difference,
        0.0,
        xerr=np.array([
            [lower_error],
            [upper_error],
        ]),
        fmt="o",
        markersize=8,
        capsize=6,
        linewidth=2,
    )

    ax.axvline(
        0.0,
        linestyle="--",
        linewidth=1.2,
    )

    ax.set_yticks([])
    ax.set_title(title)
    ax.set_xlabel(
        "Recurrent FM − Static FM\n"
        "mean trajectory energy discrepancy"
    )
    ax.grid(axis="x", alpha=0.25)

    ax.text(
        0.02,
        0.08,
        "Recurrent FM better",
        transform=ax.transAxes,
        ha="left",
        fontsize=9,
    )

    ax.text(
        0.98,
        0.08,
        "Static FM better",
        transform=ax.transAxes,
        ha="right",
        fontsize=9,
    )

fig.suptitle(
    "Paired bootstrap comparison across held-out parameter groups",
    y=1.03,
)
fig.tight_layout()

BOOTSTRAP_FIG = FIG_ROOT / "paired_group_bootstrap.png"

fig.savefig(
    BOOTSTRAP_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

BOOTSTRAP_CSV = FIG_ROOT / "paired_group_bootstrap_summary.csv"

with BOOTSTRAP_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)

    writer.writerow([
        "case",
        "contrast",
        "mean_difference",
        "ci95_low",
        "ci95_high",
        "n_groups",
    ])

    for case_name, data in [
        ("RAD", rad_bootstrap),
        ("Heat", heat_bootstrap),
    ]:
        writer.writerow([
            case_name,
            data["contrast"],
            data["mean_difference"],
            data["ci95"][0],
            data["ci95"][1],
            data["n_groups"],
        ])

print("Saved:", BOOTSTRAP_FIG)
print("Saved:", BOOTSTRAP_CSV)


# Chapter 7 — POD diagnostics and exact rank-selection verification

The final POD diagnostic must use the **same trajectory-relative metric as the code used for rank selection**:

\[
e_i(r)=
\frac{\sum_k \|u_i(t_k)-\widehat u_i^{(r)}(t_k)\|_2}
{\sum_k \|u_i(t_k)\|_2+\varepsilon}.
\]

The selected rank is the first rank for which the **training-set mean** of this quantity is at most `1%`.

This is different from computing one Frobenius norm over the complete trajectory.


In [ ]:
# ============================================================
# 7.1 — POD spectral diagnostics
# ============================================================

def pod_metadata(prepared):
    pod = prepared["pod"]

    return {
        "singular_values": np.asarray(
            as_numpy(pod["singular_values"])
        ),
        "cumulative_energy": np.asarray(
            as_numpy(pod["cumulative_energy"])
        ),
        "n_modes": int(pod["n_modes"]),
        "n_dofs": int(pod["n_dofs"]),
        "max_modes": int(pod["max_modes"]),
        "target_error": float(pod["target_error"]),
        "energy_threshold": float(pod["energy_threshold"]),
        "selection": pod["selection"],
    }

RAD_POD_META = pod_metadata(RAD_CASE["prepared"])
HEAT_POD_META = pod_metadata(HEAT_CASE["prepared"])

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8),
)

pod_cases = [
    (
        "Randomized advection-diffusion",
        RAD_POD_META,
        0,
    ),
    (
        "Stochastic heat equation",
        HEAT_POD_META,
        1,
    ),
]

for title, diag, col in pod_cases:
    n_show = min(
        diag["max_modes"],
        len(diag["singular_values"]),
    )

    mode_numbers = np.arange(1, n_show + 1)

    singular = diag["singular_values"][:n_show]

    axes[0, col].semilogy(
        mode_numbers,
        singular / singular[0],
        linewidth=2,
    )

    axes[0, col].axvline(
        diag["n_modes"],
        linestyle="--",
        linewidth=1.5,
        label=f"Selected rank r={diag['n_modes']}",
    )

    axes[0, col].set_title(title)
    axes[0, col].set_xlabel("POD mode")
    axes[0, col].set_ylabel("Normalized singular value")
    axes[0, col].grid(alpha=0.25)
    axes[0, col].legend()

    cumulative = diag["cumulative_energy"][:n_show]

    axes[1, col].plot(
        mode_numbers,
        cumulative,
        linewidth=2,
    )

    axes[1, col].axhline(
        diag["energy_threshold"],
        linestyle=":",
        linewidth=1.5,
        label="99.9% energy",
    )

    axes[1, col].axvline(
        diag["n_modes"],
        linestyle="--",
        linewidth=1.5,
        label=f"Selected rank r={diag['n_modes']}",
    )

    axes[1, col].set_xlabel("Number of retained modes")
    axes[1, col].set_ylabel("Cumulative POD energy")
    axes[1, col].grid(alpha=0.25)
    axes[1, col].legend()

fig.suptitle(
    "POD spectral diagnostics",
    y=1.01,
)
fig.tight_layout()

POD_SPECTRUM_FIG = FIG_ROOT / "pod_spectrum_energy.png"

fig.savefig(
    POD_SPECTRUM_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

print("Saved:", POD_SPECTRUM_FIG)


In [ ]:
# ============================================================
# 7.2 — Exact POD trajectory metric used by rank selection
# ============================================================

def pod_trajectory_errors_exact(
    trajectories,
    mean,
    basis,
    rank,
):
    trajectories = np.asarray(trajectories)
    mean = np.asarray(mean).reshape(-1)
    basis_rank = np.asarray(basis[:rank])

    errors = np.zeros(
        trajectories.shape[0],
        dtype=float,
    )

    for i in range(trajectories.shape[0]):
        trajectory = trajectories[i]

        coefficients = (
            trajectory - mean[None, :]
        ) @ basis_rank.T

        reconstruction = (
            mean[None, :]
            + coefficients @ basis_rank
        )

        num = np.linalg.norm(
            trajectory - reconstruction,
            axis=-1,
        )

        den = np.linalg.norm(
            trajectory,
            axis=-1,
        )

        errors[i] = (
            np.sum(num)
            / (np.sum(den) + 1e-12)
        )

    return errors

def exact_pod_selection_diagnostic(
    case_name,
    prepared,
    trajectories,
):
    pod = prepared["pod"]

    mean = np.asarray(
        as_numpy(pod["mean"])
    )
    basis = np.asarray(
        as_numpy(pod["basis"])
    )

    rank = int(pod["n_modes"])
    target = float(pod["target_error"])

    split = prepared["split"]

    split_indices = {
        "Train": as_numpy(split["train_indices"]).astype(int),
        "Validation": as_numpy(split["val_indices"]).astype(int),
        "Test": as_numpy(split["test_indices"]).astype(int),
    }

    trajectories = np.asarray(trajectories)

    print("\n" + "=" * 72)
    print(case_name)
    print("=" * 72)

    print("Selected rank:", rank)
    print("Target:", target)

    train_trajectories = trajectories[
        split_indices["Train"]
    ]

    selected_errors = pod_trajectory_errors_exact(
        train_trajectories,
        mean,
        basis,
        rank,
    )

    selected_mean = float(np.mean(selected_errors))

    print("\n=== SELECTION BOUNDARY CHECK ===")

    if rank > 1:
        previous_errors = pod_trajectory_errors_exact(
            train_trajectories,
            mean,
            basis,
            rank - 1,
        )

        previous_mean = float(np.mean(previous_errors))

        print(
            f"Rank {rank - 1} training mean = "
            f"{100 * previous_mean:.6f}%"
        )

    print(
        f"Rank {rank} training mean = "
        f"{100 * selected_mean:.6f}%"
    )

    print(
        "Selected rank satisfies target:",
        selected_mean <= target,
    )

    final_split_errors = {}

    print("\n=== FINAL-RANK TRAJECTORY ERRORS ===")

    for split_name, indices in split_indices.items():
        values = pod_trajectory_errors_exact(
            trajectories[indices],
            mean,
            basis,
            rank,
        )

        final_split_errors[split_name] = values

        print(f"\n{split_name}")
        print("  n =", len(values))
        print(f"  mean   = {100*np.mean(values):.4f}%")
        print(f"  median = {100*np.median(values):.4f}%")
        print(f"  p90    = {100*np.quantile(values, 0.90):.4f}%")
        print(f"  max    = {100*np.max(values):.4f}%")

    return {
        "rank": rank,
        "target": target,
        "errors": final_split_errors,
        "selected_train_mean": selected_mean,
    }

RAD_POD_EXACT = exact_pod_selection_diagnostic(
    "RAD POD",
    RAD_CASE["prepared"],
    RAD_CASE["dataset"][3],
)

HEAT_POD_EXACT = exact_pod_selection_diagnostic(
    "STOCHASTIC HEAT POD",
    HEAT_CASE["prepared"],
    HEAT_CASE["dataset"][3],
)


In [ ]:
# ============================================================
# 7.3 — Thesis-ready POD trajectory reconstruction boxplots
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(11, 4.5),
)

plot_cases = [
    (
        axes[0],
        "Randomized advection-diffusion",
        RAD_POD_EXACT,
    ),
    (
        axes[1],
        "Stochastic heat equation",
        HEAT_POD_EXACT,
    ),
]

for ax, title, result in plot_cases:
    plot_data = [
        100 * result["errors"][split_name]
        for split_name in [
            "Train",
            "Validation",
            "Test",
        ]
    ]

    ax.boxplot(
        plot_data,
        tick_labels=[
            "Train",
            "Validation",
            "Test",
        ],
        showmeans=True,
    )

    ax.axhline(
        100 * result["target"],
        linestyle="--",
        linewidth=1.5,
        label="1% training-mean selection target",
    )

    ax.set_title(title)
    ax.set_ylabel("Trajectory-relative POD error [%]")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)

fig.suptitle(
    "POD trajectory reconstruction accuracy",
    y=1.02,
)
fig.tight_layout()

POD_ERROR_FIG = FIG_ROOT / "pod_reconstruction_errors.png"

fig.savefig(
    POD_ERROR_FIG,
    dpi=300,
    bbox_inches="tight",
)
plt.show()
plt.close(fig)

POD_EXACT_CSV = FIG_ROOT / "pod_selection_metric_summary.csv"

with POD_EXACT_CSV.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)

    writer.writerow([
        "case",
        "selected_rank",
        "split",
        "mean",
        "median",
        "p90",
        "max",
    ])

    for case_name, result in [
        ("RAD", RAD_POD_EXACT),
        ("Heat", HEAT_POD_EXACT),
    ]:
        for split_name in [
            "Train",
            "Validation",
            "Test",
        ]:
            values = result["errors"][split_name]

            writer.writerow([
                case_name,
                result["rank"],
                split_name,
                float(np.mean(values)),
                float(np.median(values)),
                float(np.quantile(values, 0.90)),
                float(np.max(values)),
            ])

print("Saved:", POD_ERROR_FIG)
print("Saved:", POD_EXACT_CSV)


# Chapter 8 — Interpretation checklist for thesis writing

The numerical outputs produced above support the following conservative conclusions.

### RAD
- Static FM has the best primary full-trajectory energy discrepancy.
- Recurrent FM has the best increment and temporal-covariance diagnostics.
- Recurrent FM is very accurate at early times but degrades over long rollouts.
- The observed degradation is **consistent with** autoregressive transition-error accumulation / exposure bias; it does not prove causality.
- Static FM becomes better than Recurrent FM in the mean marginal discrepancy at the first sampled crossover around `t = 1.2`.

### Stochastic Heat
- Recurrent FM has the best average full-trajectory score and wins 5/6 held-out groups.
- Recurrent FM strongly improves increment and temporal-covariance diagnostics.
- Time-resolved one-time marginal discrepancies remain broadly similar across models, showing that good marginals do not imply a good path law.
- RealNVP remains best for the exact mean field.
- Recurrent FM is slightly best for the exact variance field.
- High Recurrent-FM seed variability must be reported.

### Across both benchmarks
- No single architecture is uniformly superior.
- Marginal accuracy, temporal dependence, full path accuracy and computational cost are distinct properties.


# Chapter 9 — Prepare the final figure package for Overleaf

This cell gathers all thesis figures in a flat `Images/` folder and creates `Images.zip`.

It does not modify the original figures.


In [ ]:
# ============================================================
# 9.1 — Collect all final figures required by Overleaf
# ============================================================

import zipfile

OVERLEAF_DIR = DRIVE_TESI / "overleaf_final"
OVERLEAF_IMAGES = OVERLEAF_DIR / "Images"
OVERLEAF_ZIP = OVERLEAF_DIR / "Images.zip"

OVERLEAF_IMAGES.mkdir(
    parents=True,
    exist_ok=True,
)

sources = [
    # POD diagnostics
    FIG_ROOT / "pod_spectrum_energy.png",
    FIG_ROOT / "pod_reconstruction_errors.png",

    # RAD main figures
    RAD_FIG / "rad_fixed_sample_snapshots.png",
    RAD_FIG / "rad_mean_error_maps.png",

    # Heat main figures
    HEAT_FIG / "heat_fixed_sample_spacetime.png",
    HEAT_FIG / "heat_exact_mean_profiles.png",
    HEAT_FIG / "heat_exact_std_profiles.png",

    # Common comparison figures
    FIG_ROOT / "time_resolved_marginal_energy.png",
    FIG_ROOT / "accuracy_cost_tradeoff.png",
    FIG_ROOT / "paired_group_bootstrap.png",

    # Existing group-wise comparison
    RAD_FIG / "rad_group_energy.png",
    HEAT_FIG / "heat_group_energy.png",

    # Existing probe quantiles
    RAD_FIG / "rad_probe_quantiles.png",
    HEAT_FIG / "heat_probe_quantiles.png",

    # RAD covariance figures
    RAD_FIG / "rad_covariance_reference_g5.png",
    RAD_FIG / "rad_covariance_realnvp_shared_s123_g5.png",
    RAD_FIG / "rad_covariance_flow_matching_shared_s123_g5.png",
    RAD_FIG / "rad_covariance_dynamic_s123_g5.png",

    # Heat covariance figures
    HEAT_FIG / "heat_covariance_reference_g5.png",
    HEAT_FIG / "heat_covariance_realnvp_shared_s123_g5.png",
    HEAT_FIG / "heat_covariance_flow_matching_shared_s123_g5.png",
    HEAT_FIG / "heat_covariance_dynamic_s123_g5.png",
]

missing = [
    str(path)
    for path in sources
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(
        "Missing required figures:\n"
        + "\n".join(missing)
    )

# Remove only old PNG copies in the export folder.
for old_file in OVERLEAF_IMAGES.glob("*.png"):
    old_file.unlink()

for source in sources:
    shutil.copy2(
        source,
        OVERLEAF_IMAGES / source.name,
    )

if OVERLEAF_ZIP.exists():
    OVERLEAF_ZIP.unlink()

with zipfile.ZipFile(
    OVERLEAF_ZIP,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(OVERLEAF_IMAGES.glob("*.png")):
        archive.write(
            path,
            arcname=f"Images/{path.name}",
        )

print("=== OVERLEAF IMAGE PACKAGE ===")
print("Images collected:", len(sources))
print("Folder:", OVERLEAF_IMAGES)
print("ZIP:", OVERLEAF_ZIP)
print("ZIP exists:", OVERLEAF_ZIP.is_file())
print(
    "ZIP size [MB]:",
    OVERLEAF_ZIP.stat().st_size / (1024 ** 2),
)

print("\nFiles:")
for path in sorted(OVERLEAF_IMAGES.glob("*.png")):
    print("✓", path.name)


# Chapter 10 — Final completion checklist

After a successful run, the following items should exist.

### Professor request: computed-solution visualizations
- `heat/heat_fixed_sample_spacetime.png`
- `heat/heat_exact_mean_profiles.png`
- `heat/heat_exact_std_profiles.png`
- `rad/rad_fixed_sample_snapshots.png`
- `rad/rad_mean_error_maps.png`

### Professor request: error trends over time
- `time_resolved_marginal_energy.png`
- `time_resolved_marginal_energy.csv`
- `early_late_marginal_energy_summary.csv`

### Professor request: deeper strategy comparison
- `accuracy_cost_tradeoff.png`
- `paired_group_bootstrap.png`
- corresponding CSV summaries
- pre-existing group, covariance and quantile diagnostics

### Additional benchmark
- stochastic heat equation analysis and exact-moment validation

### POD diagnostics
- `pod_spectrum_energy.png`
- `pod_reconstruction_errors.png`
- `pod_selection_metric_summary.csv`

### Overleaf
- `overleaf_final/Images.zip`

The **recent-state-of-the-art literature review is intentionally handled in the thesis LaTeX source**, not in this numerical notebook.
